In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_excel("Тестовое_задание_Data_аналитик_ЦО_2025_2.xlsx", sheet_name="Данные для задачи 3",engine="calamine")
data

,Бренд AX 1,Неделя,Выручка,Маржинальность
0,Shiseido,14,1.353686e+09,0.167824
1,Clarins,14,1.329934e+09,0.091348
2,Shiseido,15,1.267702e+09,0.169212
3,Clarins,15,9.747119e+08,0.145904
4,Clinique,15,4.982012e+08,0.140136
...,...,...,...,...
99,Giorgio Armani,15,1.630626e+07,0.141214
100,Proraso,14,1.617463e+07,0.220514
101,Givenchy,15,1.535237e+07,0.112949
102,Grown Alchemist,14,1.528858e+07,0.203719


In [3]:
# Подготовка и проверка данных.
# Названия столбцов сохраняются без изменений.
def to_number(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

data = data.copy()

data["Неделя"] = to_number(data["Неделя"])
data["Выручка"] = to_number(data["Выручка"])
data["Маржинальность"] = to_number(data["Маржинальность"])

invalid_rows = data.loc[
    data["Бренд AX 1"].isna()
    | data["Неделя"].isna()
    | data["Выручка"].isna()
    | data["Маржинальность"].isna()
].copy()

print("Некорректных строк:", len(invalid_rows))
display(invalid_rows.head())

data = data.drop(index=invalid_rows.index).copy()

weeks = sorted(data["Неделя"].unique())

if len(weeks) < 2:
    raise ValueError("Для сравнения нужны данные минимум за две недели.")

prelast_week = weeks[-2]
last_week = weeks[-1]

print(f"Позапрошлая неделя: {prelast_week}")
print(f"Прошлая неделя: {last_week}")

Некорректных строк: 0


,Бренд AX 1,Неделя,Выручка,Маржинальность


Позапрошлая неделя: 14
Прошлая неделя: 15


In [4]:
# Агрегация по брендам и неделям.
# Валовая прибыль = Выручка * Маржинальность.
analysis = data.copy()
analysis["Валовая прибыль"] = analysis["Выручка"] * analysis["Маржинальность"]

brand_week = analysis.groupby(["Бренд AX 1", "Неделя"], as_index=False).agg({"Выручка": "sum", "Валовая прибыль": "sum"})

brand_week["Маржинальность"] = brand_week["Валовая прибыль"] / brand_week["Выручка"]
brand_week.sort_values(["Неделя", "Выручка"], ascending=[True, False])

,Бренд AX 1,Неделя,Выручка,Валовая прибыль,Маржинальность
88,Shiseido,14,1353685520.769999,227180372.469722,0.167824
23,Clarins,14,1329934371.92,121486276.257183,0.091348
27,Clinique,14,485396784.14,82392996.649828,0.169744
38,Estee Lauder,14,328740954.37,35615690.980468,0.10834
4,BABOR,14,309282334.73,45687429.722349,0.147721
...,...,...,...,...,...
7,Biothal,15,17949429.01,2542770.509786,0.141663
14,CHOLLEY,15,16931694.37,1933309.453801,0.114183
44,Giorgio Armani,15,16306263.34,2302671.306327,0.141214
46,Givenchy,15,15352372.07,1734030.108102,0.112949


### Промежуточный вывод: валовая прибыль
- валовая прибыль = выручка * маржинальность
- маржа бренда за неделю считается как прибыль / выручка

In [5]:
# Сравнение прошлой и позапрошлой недель по брендам.
prelast_week = (
    brand_week.loc[brand_week["Неделя"].eq(prelast_week), ["Бренд AX 1", "Выручка", "Валовая прибыль", "Маржинальность"]]
    .set_index("Бренд AX 1")
    .add_suffix(" — позапрошлая неделя")
)
last_week = (
    brand_week.loc[brand_week["Неделя"].eq(last_week),["Бренд AX 1", "Выручка", "Валовая прибыль", "Маржинальность"]]
    .set_index("Бренд AX 1")
    .add_suffix(" — прошлая неделя")
)

comparison = prelast_week.join(last_week, how="outer").fillna(0)
comparison["Изменение выручки"] = comparison["Выручка — прошлая неделя"] - comparison["Выручка — позапрошлая неделя"]
comparison["Изменение выручки, %"] = np.where(comparison["Выручка — позапрошлая неделя"] > 0,
                                              comparison["Изменение выручки"]
                                              / comparison["Выручка — позапрошлая неделя"],
                                              np.nan)
comparison["Изменение валовой прибыли"] = comparison["Валовая прибыль — прошлая неделя"] - comparison["Валовая прибыль — позапрошлая неделя"]

comparison["Изменение маржинальности, п.п."] = (comparison["Маржинальность — прошлая неделя"] - comparison["Маржинальность — позапрошлая неделя"]) * 100
comparison.sort_values("Изменение валовой прибыли")

,Выручка — позапрошлая неделя,Валовая прибыль — позапрошлая неделя,Маржинальность — позапрошлая неделя,Выручка — прошлая неделя,Валовая прибыль — прошлая неделя,Маржинальность — прошлая неделя,Изменение выручки,"Изменение выручки, %",Изменение валовой прибыли,"Изменение маржинальности, п.п."
Бренд AX 1,,,,,,,,,,
Shiseido,1353685520.769999,227180372.469722,0.167824,1267702457.570001,214510287.572773,0.169212,-85983063.199998,-0.063518,-12670084.896948,0.138826
Clinique,485396784.14,82392996.649828,0.169744,498201172.67,69815982.710062,0.140136,12804388.53,0.026379,-12577013.939766,-2.960747
Sisley,301387817.96,46849781.622212,0.155447,206654314.05,34514352.381429,0.167015,-94733503.91,-0.314324,-12335429.240783,1.156808
SENSAI,161624333.42,23815834.310423,0.147353,125798038.82,18234682.539827,0.144952,-35826294.6,-0.221664,-5581151.770596,-0.240098
Dr Barbara Sturm,41725673.52,9318365.229859,0.223325,27390688.11,5333922.580792,0.194735,-14334985.41,-0.343553,-3984442.649067,-2.858961
CHANEL,236703280.29,20775902.835685,0.087772,217319320.17,16796437.583068,0.077289,-19383960.12,-0.081891,-3979465.252617,-1.048272
Helena Rubinstein,48107733.58,6253985.210287,0.13,35725576.31,2879631.996359,0.080604,-12382157.27,-0.257384,-3374353.213928,-4.939537
FILORGA,60160904.53,12589842.044606,0.209269,43946338.03,9418020.913063,0.214307,-16214566.5,-0.269520,-3171821.131543,0.50378
Grown Alchemist,15288583.5,3114570.596268,0.203719,0.0,0.0,0.0,-15288583.5,-1.000000,-3114570.596268,-20.371872


### Промежуточный вывод: сравнение недель
- для каждого бренда посчитали изменение выручки, прибыли и маржи
- бренды, которых не было в одной из недель, считаются с нулём

In [6]:
# Декомпозиция изменения валовой прибыли по брендам.
# Влияние выручки: насколько изменилась бы валовая прибыль при старой маржинальности.
# Влияние маржинальности: насколько изменилась бы валовая прибыль при прежней выручке.
# Взаимодействие: совместный эффект изменения выручки и маржинальности.

comparison["Влияние выручки на изменение валовой прибыли"] = (comparison["Выручка — прошлая неделя"] - comparison["Выручка — позапрошлая неделя"]) * comparison["Маржинальность — позапрошлая неделя"]
comparison["Влияние маржинальности на изменение валовой прибыли"] = comparison["Выручка — позапрошлая неделя"]* (comparison["Маржинальность — прошлая неделя"]- comparison["Маржинальность — позапрошлая неделя"])

comparison["Взаимодействие выручки и маржинальности"] = (comparison["Выручка — прошлая неделя"] - comparison["Выручка — позапрошлая неделя"]) * (comparison["Маржинальность — прошлая неделя"] - comparison["Маржинальность — позапрошлая неделя"])
comparison["Проверка декомпозиции"] = comparison["Влияние выручки на изменение валовой прибыли"] + comparison["Влияние маржинальности на изменение валовой прибыли"] + comparison["Взаимодействие выручки и маржинальности"]
comparison["Отклонение проверки"] = comparison["Изменение валовой прибыли"] - comparison["Проверка декомпозиции"]

drivers = comparison[[
        "Изменение выручки",
        "Изменение валовой прибыли",
        "Изменение маржинальности, п.п.",
        "Влияние выручки на изменение валовой прибыли",
        "Влияние маржинальности на изменение валовой прибыли",
        "Взаимодействие выручки и маржинальности",
        "Отклонение проверки",
    ]].sort_values("Изменение валовой прибыли")
drivers

,Изменение выручки,Изменение валовой прибыли,"Изменение маржинальности, п.п.",Влияние выручки на изменение валовой прибыли,Влияние маржинальности на изменение валовой прибыли,Взаимодействие выручки и маржинальности,Отклонение проверки
Бренд AX 1,,,,,,,
Shiseido,-85983063.199998,-12670084.896948,0.138826,-14429986.89441,1879268.938631,-119366.94117,0.0
Clinique,12804388.53,-12577013.939766,-2.960747,2173462.980651,-14371371.354528,-379105.565889,-0.0
Sisley,-94733503.91,-12335429.240783,1.156808,-14726023.103825,3486478.718449,-1095884.855406,-0.0
SENSAI,-35826294.6,-5581151.770596,-0.240098,-5279112.854454,-388057.230036,86018.313893,-0.0
Dr Barbara Sturm,-14334985.41,-3984442.649067,-2.858961,-3201353.467693,-1192920.871058,409831.689684,-0.0
CHANEL,-19383960.12,-3979465.252617,-1.048272,-1701367.516033,-2481294.376629,203196.640045,-0.0
Helena Rubinstein,-12382157.27,-3374353.213928,-4.939537,-1609675.257498,-2376299.160181,611621.203751,-0.0
FILORGA,-16214566.5,-3171821.131543,0.50378,-3393214.125545,303078.78592,-81685.791918,-0.0
Grown Alchemist,-15288583.5,-3114570.596268,-20.371872,-3114570.596268,-3114570.596268,3114570.596268,0.0


### Промежуточный вывод: что изменило прибыль
- изменение прибыли разложено на эффект выручки и эффект маржи
- в сумме части сходятся с итоговым изменением, значит расчёт верный

In [7]:
data["Неделя"] = pd.to_numeric(data["Неделя"], errors="coerce")

weeks = sorted(data["Неделя"].dropna().unique())

prelast_week = weeks[-2]
last_week = weeks[-1]

print(prelast_week, type(prelast_week))
print(last_week, type(last_week))

14 <class 'numpy.int64'>
15 <class 'numpy.int64'>


In [8]:
# Общая маржинальность и её разложение: изменение маржи брендов и изменение структуры выручки (микса).
overall = brand_week.groupby("Неделя", as_index=False).agg({"Выручка": "sum","Валовая прибыль": "sum"})
overall["Маржинальность"] = overall["Валовая прибыль"] / overall["Выручка"]
overall_previous = overall.loc[overall["Неделя"] == prelast_week].iloc[0]
overall_current = overall.loc[overall["Неделя"].eq(last_week)].iloc[0]
overall_margin_change_pp = (overall_current["Маржинальность"] - overall_previous["Маржинальность"]) * 100

print("Общая выручка, позапрошлая неделя:", f"{overall_previous['Выручка']:,.0f}")
print("Общая выручка, прошлая неделя:", f"{overall_current['Выручка']:,.0f}")
print("Изменение выручки:", f"{overall_current['Выручка'] - overall_previous['Выручка']:,.0f}")
print("Общая маржинальность, позапрошлая неделя:", f"{overall_previous['Маржинальность']:.2%}")
print("Общая маржинальность, прошлая неделя:", f"{overall_current['Маржинальность']:.2%}")
print("Изменение общей маржинальности, п.п.:", f"{overall_margin_change_pp:+.2f}")

Общая выручка, позапрошлая неделя: 8,464,188,705
Общая выручка, прошлая неделя: 8,207,907,632
Изменение выручки: -256,281,073
Общая маржинальность, позапрошлая неделя: 14.63%
Общая маржинальность, прошлая неделя: 15.45%
Изменение общей маржинальности, п.п.: +0.82


### Промежуточный вывод: итоги по всему бизнесу
- выручка снизилась с 8 464 до 8 208 млн руб. (-256 млн, около -3%)
- маржа выросла с 14,63% до 15,45% (+0,82 п.п.)
- валовая прибыль выросла с 1 238 до 1 268 млн руб. (+30 млн), то есть мы продали меньше, но заработали больше

In [9]:
comparison["Доля выручки — позапрошлая неделя"] = np.where(overall_previous["Выручка"] > 0,
                                                           comparison["Выручка — позапрошлая неделя"] / overall_previous["Выручка"],
                                                           0)
comparison["Доля выручки — прошлая неделя"] = np.where(overall_current["Выручка"] > 0,
                                                       comparison["Выручка — прошлая неделя"] / overall_current["Выручка"],
                                                       0)

comparison["Вклад изменения маржинальности бренда, п.п."] = ((comparison["Маржинальность — прошлая неделя"] - comparison["Маржинальность — позапрошлая неделя"])
    * (comparison["Доля выручки — прошлая неделя"] + comparison["Доля выручки — позапрошлая неделя"]) / 2 * 100
)

comparison["Вклад изменения микса брендов, п.п."] = ((comparison["Доля выручки — прошлая неделя"] - comparison["Доля выручки — позапрошлая неделя"]) 
    * (comparison["Маржинальность — прошлая неделя"] + comparison["Маржинальность — позапрошлая неделя"])/ 2* 100
)

margin_drivers = comparison[[
        "Маржинальность — позапрошлая неделя",
        "Маржинальность — прошлая неделя",
        "Изменение маржинальности, п.п.",
        "Вклад изменения маржинальности бренда, п.п.",
        "Вклад изменения микса брендов, п.п."]
].sort_values("Вклад изменения маржинальности бренда, п.п.")
print(
    "Проверка разложения общей маржинальности, п.п.:",
    f"{(
        comparison['Вклад изменения маржинальности бренда, п.п.'].sum()
        + comparison['Вклад изменения микса брендов, п.п.'].sum()
    ):.2f}"
)
margin_drivers

Проверка разложения общей маржинальности, п.п.: 0.82


,Маржинальность — позапрошлая неделя,Маржинальность — прошлая неделя,"Изменение маржинальности, п.п.","Вклад изменения маржинальности бренда, п.п.","Вклад изменения микса брендов, п.п."
Бренд AX 1,,,,,
Clinique,0.169744,0.140136,-2.960747,-0.17475,0.051914
Lancaster,0.144102,0.099229,-4.487344,-0.055128,0.069268
Caudalie,0.1905,0.167459,-2.304126,-0.054818,0.108444
Ahava,0.124377,0.081975,-4.240198,-0.030019,0.037821
CHANEL,0.087772,0.077289,-1.048272,-0.028535,-0.012284
Payot,0.21654,0.209797,-0.67424,-0.025708,0.286572
Helena Rubinstein,0.13,0.080604,-4.939537,-0.024787,-0.014017
FOREO,0.211262,0.1853,-2.596218,-0.023571,0.009543
Grown Alchemist,0.203719,0.0,-20.371872,-0.018399,-0.018399


### Промежуточный вывод: почему выросла маржа
- из +0,82 п.п. около +0,54 п.п. дал рост маржи внутри брендов, и около +0,28 п.п. смена структуры продаж
- главный бренд роста маржи - Clarins: маржа выросла с 9,1% до 14,6% и дала +0,75 п.п. из 0,82
- при этом выручка Clarins упала на 355 млн руб. (с 1 330 до 975 млн), возможно, закончилась акция или скидка
- доля брендов с высокой маржой выросла: Payot (+102 млн), Valmont (+76 млн), La Mer (+46 млн)
- маржа снизилась у 30 брендов из 55, сильнее всего у Clinique (17,0% -> 14,0%) и Caudalie (около 19% -> 16,8%)

In [ ]:
# Короткая аналитическая записка
def млн(значение):
    return f"{значение / 1_000_000:,.1f}".replace(",", " ")

def список_брендов(таблица, столбец):
    результат = []
    for бренд, значение in таблица[столбец].items():
        результат.append(f"{бренд} ({млн(значение)} млн руб.)")

    return ", ".join(результат)


# Бренды с самым сильным падением и ростом выручки
падение_выручки = comparison.nsmallest(3, "Изменение выручки")
рост_выручки = comparison.nlargest(3, "Изменение выручки")

# Бренды, которые сильнее всего ухудшили общую маржинальность
ухудшение_маржи = comparison.nsmallest(3,"Вклад изменения маржинальности бренда, п.п.")

# Бренды, которые сильнее всего подняли общую маржинальность
улучшение_маржи = comparison.nlargest(3, "Вклад изменения маржинальности бренда, п.п.")

# Из чего сложился рост общей маржи: маржа внутри брендов и структура продаж
вклад_маржи_брендов = comparison["Вклад изменения маржинальности бренда, п.п."].sum()
вклад_структуры = comparison["Вклад изменения микса брендов, п.п."].sum()

# Бренд с наибольшим отрицательным эффектом изменения маржинальности
бренд_для_быстрого_действия = comparison["Влияние маржинальности на изменение валовой прибыли"].idxmin()
эффект_быстрого_действия = comparison.loc[бренд_для_быстрого_действия, "Влияние маржинальности на изменение валовой прибыли"]
изменение_выручки = (overall_current["Выручка"] - overall_previous["Выручка"])
направление_выручки = ("выросла" if изменение_выручки >= 0 else "снизилась")
направление_маржи = ("выросла" if overall_margin_change_pp >= 0 else "снизилась")

In [11]:
note = f"""
За прошлую неделю ({int(last_week)}) выручка составила
{млн(overall_current["Выручка"])} млн руб.
По сравнению с позапрошлой неделей выручка {направление_выручки} на
{млн(abs(изменение_выручки))} млн руб.
Маржинальность {направление_маржи} с
{overall_previous["Маржинальность"]:.1%} до
{overall_current["Маржинальность"]:.1%}
({overall_margin_change_pp:+.2f} п.п.).
Из них {вклад_маржи_брендов:+.2f} п.п. дало изменение маржи внутри брендов
и {вклад_структуры:+.2f} п.п. дала смена структуры продаж.
Больше всего выручка снизилась по брендам:
{список_брендов(падение_выручки, "Изменение выручки")}.
Больше всего выручка выросла по брендам:
{список_брендов(рост_выручки, "Изменение выручки")}.
Сильнее всего маржинальность подняли:
{", ".join([
    f"{бренд} ({значение:+.2f} п.п.)"
    for бренд, значение in улучшение_маржи[
        "Вклад изменения маржинальности бренда, п.п."
    ].items()
])}.
У {улучшение_маржи.index[0]} выручка при этом изменилась на
{млн(comparison.loc[улучшение_маржи.index[0], "Изменение выручки"])} млн руб.
Стоит проверить, не закончилась ли акция: тогда рост маржи может быть временным.
Сильнее всего маржинальность снизили:
{", ".join([
    f"{бренд} ({значение:+.2f} п.п.)"
    for бренд, значение in ухудшение_маржи[
        "Вклад изменения маржинальности бренда, п.п."
    ].items()
])}.
Что можно сделать быстро: проверить скидки по бренду
{бренд_для_быстрого_действия}. Это бренд с самым заметным
отрицательным влиянием на валовую прибыль
({млн(эффект_быстрого_действия)} млн руб. за неделю).
"""
print(note)


За прошлую неделю (15) выручка составила
8 207.9 млн руб.
По сравнению с позапрошлой неделей выручка снизилась на
256.3 млн руб.
Маржинальность выросла с
14.6% до
15.5%
(+0.82 п.п.).
Из них +0.54 п.п. дало изменение маржи внутри брендов
и +0.28 п.п. дала смена структуры продаж.
Больше всего выручка снизилась по брендам:
Clarins (-355.2 млн руб.), Sisley (-94.7 млн руб.), Shiseido (-86.0 млн руб.).
Больше всего выручка выросла по брендам:
Payot (102.3 млн руб.), Valmont (75.7 млн руб.), La Mer (46.0 млн руб.).
Сильнее всего маржинальность подняли:
Clarins (+0.75 п.п.), Lancome (+0.07 п.п.), La Prairie (+0.05 п.п.).
У Clarins выручка при этом изменилась на
-355.2 млн руб.
Стоит проверить, не закончилась ли акция: тогда рост маржи может быть временным.
Сильнее всего маржинальность снизили:
Clinique (-0.17 п.п.), Lancaster (-0.06 п.п.), Caudalie (-0.05 п.п.).
Что можно сделать быстро: проверить скидки по бренду
Clinique. Это бренд с самым заметным
отрицательным влиянием на валовую прибыль

## Итог по заданию 3
- выручка упала на 3%, но маржа выросла на 0,82 п.п., и прибыль стала больше на 30 млн руб.
- рост маржи почти целиком даёт Clarins, и вопрос в том, надолго ли это: выручка бренда упала на четверть, и маржа могла вырасти из-за конца скидок
- Clinique потеряла 14 млн руб. прибыли за неделю из-за падения маржи, при этом выручка бренда даже выросла
- быстрое действие: проверить скидки и цены по Clinique и вернуть маржу прошлой недели, это даст около 14 млн руб. прибыли в неделю
- следом по размеру потерь Caudalie (около 4 млн) и Lancaster (около 3,6 млн)
- в данных нет скидок и цен, поэтому причины выше - лишь *гипотезы*, и их надо *проверять*